In [1]:
import os
import json
import time
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown, HTML

load_dotenv()

MODEL = "gpt-4o-mini"
client = OpenAI()

print(f"\u2705 Client ready | model : {MODEL}")



✅ Client ready | model : gpt-4o-mini


In [2]:
response = client.chat.completions.create(
    model= MODEL,
    messages=[
        {"role": "system", "content": "You are a coding assitant, generate clean code"},
        {"role": "user", "content": "Generate me a efficient code in linked list reversal"}
    ]
)

In [3]:
#  HELPER FUNCTIONS

def chat(messages, max_tokens=150, **kwargs):
    """Send messages to OpenAI and display the response."""
    start = time.time()
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=max_tokens,
        **kwargs
    )
    elapsed = time.time() - start
    content = response.choices[0].message.content
    if content:
        display(Markdown(content))
    print(f"\n\u23f1\ufe0f {elapsed:.2f}s | Tokens: {response.usage.prompt_tokens}+{response.usage.completion_tokens}={response.usage.total_tokens}")
    return response


def show_messages(messages):
    """Pretty-print the message list being sent (handles both dicts and OpenAI objects)."""
    colors = {"system": "#e74c3c", "user": "#3498db", "assistant": "#2ecc71", "tool": "#f39c12"}
    html = ""
    for msg in messages:
        if isinstance(msg, dict): # isinstance() : checks the type of an object : isinstance(object, type) i.e true or false.
            role = msg.get("role", "unknown")
            content = msg.get("content", "")
        else:
            role = getattr(msg, "role", "unknown")
            content = getattr(msg, "content", None)

        if not content:
            tool_calls = msg.get("tool_calls", None) if isinstance(msg, dict) else getattr(msg, "tool_calls", None)
            if tool_calls:
                content = ", ".join(f"{tc.function.name}({tc.function.arguments})" for tc in tool_calls)
                content = f"[tool_calls] {content}"
            else:
                content = "(empty)"

        if len(str(content)) > 200:
            content = str(content)[:200] + "..."

        color = colors.get(role, "#888")
        html += (
            f'<div style="margin:6px 0;padding:8px 12px;border-left:4px solid {color};'
            f'background:#1e1e1e;border-radius:4px;">'
            f'<strong style="color:{color};text-transform:uppercase;">{role}</strong>'
            f'<br><span style="color:#ccc;">{content}</span></div>'
        )
    display(HTML(html))


print("\u2705 Helpers loaded")

✅ Helpers loaded


In [4]:
# Step 1: Define the ACTUAL function your code will execute
def get_weather(location:str, unit:str="celsius"):
     # In production, this would call a real weather API
    weather_data = {
        "bangkok" : {"temp" : 29, "condition": "Cloudy"},
        "paris"   : {"temp" : 27, "condition" : "Humid"},
        "new york": {"temp" : 30, "condition" : "Sunny"},
    }
    data = weather_data.get(location.lower(), {"temp": 20, "condition": "None"})
    if unit == "fahrenheit":
        data["temp"] = round(data["temp"] * 9/5 + 32)
    data["unit"] = unit
    data["location"] = location
    return data

# Step 2: Describe it to the model (tool schema)
weather_tool = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current weather for a given location.",
        "parameters":{
            "type": "object",
            "properties": {
                "location": {
                    "type" : "string",
                    "description": "city name, e.g. 'New York'"
                },
                "unit": {
                    "type": "string",
                    "enum": ["celsius", "fahrenheit"],
                    "description": "Temperature unit"
                }
            },
            "required":["location"]
        }
     }
}
print(json.dumps(weather_tool, indent=2))

{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "Get the current weather for a given location.",
    "parameters": {
      "type": "object",
      "properties": {
        "location": {
          "type": "string",
          "description": "city name, e.g. 'New York'"
        },
        "unit": {
          "type": "string",
          "enum": [
            "celsius",
            "fahrenheit"
          ],
          "description": "Temperature unit"
        }
      },
      "required": [
        "location"
      ]
    }
  }
}


In [5]:
# Step 3: Send the request
messages = [
    {"role": "user", "content": "What's the weather in Paris?"}
]
show_messages(messages)
response = chat(messages, max_tokens=100, tools = [weather_tool])

# Step 4: Check if the model wants to call a function
choice = response.choices[0]
print(f"finsh reason : {choice.finish_reason}")

if choice.finish_reason == "tool_calls":
    tool_call = choice.message.tool_calls[0]
    print(f"  Function    :{tool_call.function.name}")
    print(f"  Arguments   :{tool_call.function.arguments}")
    print(f"  Call ID     :{tool_call.id}")


⏱️ 1.22s | Tokens: 77+14=91
finsh reason : tool_calls
  Function    :get_weather
  Arguments   :{"location":"Paris"}
  Call ID     :call_xezc6fRSbuUn0GU3GetkZYvI


In [6]:
# Step 5: Execute the function with the model's arguments
args = json.loads(tool_call.function.arguments)
result = get_weather(**args)
print(f"Function result {result}")

# Step 6: Send the result bck to the model
messages.append(choice.message)
messages.append({
    "role": "tool",
    "tool_call_id" : tool_call.id,
    "content": json.dumps(result)
})

# Model generates final response using the function result
show_messages(messages)
final_response = chat(messages, max_tokens=70)

Function result {'temp': 27, 'condition': 'Humid', 'unit': 'celsius', 'location': 'Paris'}


The current weather in Paris is 27°C and humid.


⏱️ 1.23s | Tokens: 61+12=73


### Experiment 1D: `tool_choice` — Controlling When Functions Are Called

In [7]:
# tool_choice = none
messages = [
    {"role": "user", "content": "What's the weather in bangkok?"}
]
show_messages(messages)
response = chat(messages, max_tokens=50, tools =[weather_tool], tool_choice = "none")

# tool_choice="required" — model MUST call a function
messages2= [
    {"role": "user", "content": "Write a poem in 2 lines?"}
]
show_messages(messages2)
response2 = chat(messages2, max_tokens=50, tools =[weather_tool], tool_choice = "required")
tc = response2.choices[0].message.tool_calls[0]
print(f"Function name : {tc.function.name}")
print(f"Aguments : {tc.function.arguments}")

Which temperature unit would you like the weather in: Celsius or Fahrenheit?


⏱️ 1.54s | Tokens: 79+14=93



⏱️ 0.82s | Tokens: 79+15=94
Function name : get_weather
Aguments : {"location":"New York"}


### Experiment 1E: Multiple Tools — Model Picks the Right One

In [8]:
# Define a second function
def calculate(expression:str) -> str:
    try:
        result = eval(expression, {"__builtin__": {}}, {})
        return str(result)
    except Exception as e:
        print(f"Error : {e}")

calc_tool = {
    "type": "function",
    "function": {
        "name": "calculate",
        "description": "Evaluate a mathematical expression and return the result.",
        "parameters": {
            "type": "object",
            "properties": {
                "expresssion": {
                    "type": "string",
                    "description": "Math expression, e.g. '2 * 3 + 4'"
                }
            },
            "required": ["expression"]
        }
    }
}

tools = [weather_tool, calc_tool]

# Test with different queries — model picks the right tool
queries = [
    "What's the weather in New York?",
    "What is 15 * 23 + 7?",
    "Hello, how are you?"  # no tool needed
]
for q in queries:
    messages = [{"role": "user", "content": q}]
    show_messages(messages)
    resp= chat(messages, max_tokens=50, tools = tools)

    choice = resp.choices[0]
    if choice.finish_reason == "tool_calls":
        for tc in choice.message.tool_calls:
            print(f"  \u2192 Calls: {tc.function.name}({tc.function.arguments})")
    else:
        print(f"  \u2192 Direct response (no tool needed)")


⏱️ 0.92s | Tokens: 118+15=133
  → Calls: get_weather({"location":"New York"})



⏱️ 0.98s | Tokens: 122+21=143
  → Calls: calculate({"expresssion":"15 * 23 + 7"})


I'm just a program, but I'm here and ready to help you! How can I assist you today?


⏱️ 1.11s | Tokens: 117+22=139
  → Direct response (no tool needed)


### Experiment 1F: Parallel Tool Calls

In [9]:
# Ask about weather in two cities — model may call get_weather twice in parallel
messages = [{"role": "user", "content": "What's the weather in Paris and Bangkok"}]
show_messages(messages)
response = chat(messages, max_tokens=50, tools = [weather_tool])

choice = response.choices[0]
if choice.finish_reason == "tool_calls":
    print(f"Model made {len(choice.message.tool_calls)}, parellel tool calls")

    # Execute all functions
    messages.append(choice.message)

    available_functions = {"get_weather" : get_weather, "calculate": calculate}

    for tc in choice.message.tool_calls:
        fn = available_functions[tc.function.name]
        args = json.loads(tc.function.arguments) # str '{"location":"Paris"}' into {"location":"Paris"} dict
        result = fn(**args) # get_weather(location = "Tokyo") : Upacks dict into kwargs
        print(f"{tc.function.name}({args})->{result}")
        """ 
        Expected output
        get_weather({"location":"Paris"}) ->
        {'temp':27, 'condition':'Humid','unit':'celsius', 'location':'Paris'}
        """
        messages.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": json.dumps(result)
        })

    # get the final response
    print("Final response with both results")
    show_messages(messages)        
    final = chat(messages, max_tokens=100)


⏱️ 1.46s | Tokens: 78+45=123
Model made 2, parellel tool calls
get_weather({'location': 'Paris'})->{'temp': 27, 'condition': 'Humid', 'unit': 'celsius', 'location': 'Paris'}
get_weather({'location': 'Bangkok'})->{'temp': 29, 'condition': 'Cloudy', 'unit': 'celsius', 'location': 'Bangkok'}
Final response with both results


The current weather is as follows:

- **Paris**: 27°C, Humid
- **Bangkok**: 29°C, Cloudy


⏱️ 1.35s | Tokens: 124+31=155


## 2. Structured Outputs — Guaranteed JSON Schema

### Method 1: JSON Mode (Simple, Less Strict)

In [10]:
messages = [
    {"role": "system", "content": "Extract info as JSON with keys: name, topic, key_points (list)."},
    {"role": "user", "content": "Marie Curie discovered radioactivity, won two Nobel Prizes, and pioneered radiation therapy."}
]
show_messages(messages)
response = chat(messages, max_tokens=100, response_format = {"type": "json_object"})

result = response.choices[0].message.content
try:
    parsed = json.loads(result)
    print(f"Valid JSON Keys : {list(parsed.keys())}")
except json.JSONDecodeError :
    print("Unvalid JSON")

{
  "name": "Marie Curie",
  "topic": "Contributions to Science",
  "key_points": [
    "Discovered radioactivity",
    "Won two Nobel Prizes",
    "Pioneered radiation therapy"
  ]
}


⏱️ 1.28s | Tokens: 45+52=97
Valid JSON Keys : ['name', 'topic', 'key_points']


### Method 2: Structured Outputs with Pydantic (Strict, Recommended)

In [11]:
from pydantic import BaseModel

class PersonInfo(BaseModel):
    name : str
    topic : str
    key_points : list[str]

messages = [
    {"role": "system", "content": "Extract structured information from the text."},
    {"role": "user", "content": "Alan Turing invented the concept of the Turing machine, broke the Enigma code, and laid the foundations of computer science."}
]

show_messages(messages)
response = client.chat.completions.parse(
    model = MODEL,
    messages=messages,
    response_format=PersonInfo,
    max_tokens=120
)
result = response.choices[0].message.parsed # Parsed gives PersonInfo(...) - An actual Pyhton object.
print(result)

print(f"   Type:       {type(result).__name__}") # type(result) = <class '__main__.PersonInfo'>, adding .__name__ extracts only PersonInfo

print(type(PersonInfo))
print(f"\u2705 Parsed into Pydantic object:")
print(f"   Name:       {result.name}")
print(f"   Topic:      {result.topic}")
print(f"   Key Points: {result.key_points}")

print(f"\n\u23f1\ufe0f Tokens: {response.usage.prompt_tokens}+{response.usage.completion_tokens}={response.usage.total_tokens}")


name='Alan Turing' topic='Contributions to Computer Science' key_points=['Invented the concept of the Turing machine', 'Broke the Enigma code', 'Laid the foundations of computer science']
   Type:       PersonInfo
<class 'pydantic._internal._model_construction.ModelMetaclass'>
✅ Parsed into Pydantic object:
   Name:       Alan Turing
   Topic:      Contributions to Computer Science
   Key Points: ['Invented the concept of the Turing machine', 'Broke the Enigma code', 'Laid the foundations of computer science']

⏱️ Tokens: 110+44=154


## 3. Combining Function Calling + Structured Outputs

In [17]:
# A complete tool-calling loop with structured final output

class WeatherReport(BaseModel):
    location : str
    temperature : int
    condition : str
    recommendation : str

# Step 1: Define the ACTUAL function your code will execute
def get_weather(location:str, unit:str="celsius"):
     # In production, this would call a real weather API
    weather_data = {
        "bangkok" : {"temp" : 29, "condition": "Cloudy"},
        "paris"   : {"temp" : 27, "condition" : "Humid"},
        "new york": {"temp" : 30, "condition" : "Sunny"},
    }
    data = weather_data.get(location.lower(), {"temp": 20, "condition": "None"})
    if unit == "fahrenheit":
        data["temp"] = round(data["temp"] * 9/5 + 32)
    data["unit"] = unit
    data["location"] = location
    return data

# Step 2: Describe it to the model (tool schema)
weather_tool = {
    "type": "function",
    "function" : {
        "name": "get_weather",
        "description": "Get the current weather for a given location",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City name, e.g. 'New York'"
                },
                "unit": {
                    "type":"string",
                    "enum": ["celsius" or "fahrenheit"],
                    "description": "Temperature Unit"
                }
            },
            "required": ["location"]
        }
        
    }
}
# Step 1: Ask with tools
messages = [
    {"role": "system", "content": "You are a weather assistant. Use the weather tool, then summarize."},
    {"role": "user", "content": "What's the weather in New York?"}
]

print("Step 1: Send request with tools")
show_messages(messages)
resp = chat(messages, max_tokens=100, tools = [weather_tool])

# Step 2: Execute tool call
if resp.choices[0].finish_reason == "tool_calls":
    tc = resp.choices[0].message.tool_calls[0]
    args = json.loads(tc.function.arguments)
    result = get_weather(**args)

    print(result)

    print(f"\n{'=' * 60}")
    print(f"Step 2: Execute function \u2192 {result}")
    print(f"{'=' * 60}")

    messages.append(resp.choices[0].message)
    messages.append({"role": "tool","tool_call_id": tc.id, "content": json.dumps(result)})

    # Step 3: Get structured final response
    print(f"\n{'=' * 60}")
    print("Step 3: Get structured final response")
    print(f"{'=' * 60}")
    show_messages(messages)

    final_response = client.beta.chat.completions.parse(
        model=MODEL, messages=messages,
        response_format=WeatherReport, max_tokens=100
    )

    report = final_response.choices[0].message.parsed
    print(f"\u2705 Structured Report:")
    print(f"   Location:       {report.location}")
    print(f"   Temperature:    {report.temperature}")
    print(f"   Condition:      {report.condition}")
    print(f"   Recommendation: {report.recommendation}")
    print(f"\n\u23f1\ufe0f Tokens: {final_response.usage.prompt_tokens}+{final_response.usage.completion_tokens}={final_response.usage.total_tokens}")

Step 1: Send request with tools



⏱️ 2.23s | Tokens: 87+20=107
{'temp': 30, 'condition': 'Sunny', 'unit': 'celsius', 'location': 'New York'}

Step 2: Execute function → {'temp': 30, 'condition': 'Sunny', 'unit': 'celsius', 'location': 'New York'}

Step 3: Get structured final response


✅ Structured Report:
   Location:       New York
   Temperature:    30
   Condition:      Sunny
   Recommendation: It's a great day to be outdoors! Stay hydrated and wear sunscreen.

⏱️ Tokens: 155+32=187
